# Day 2: Spectral Methods and Advanced GNN Architectures

**Geometric Deep Learning Workshop**

---

## Learning Objectives

1. Compute and interpret the Graph Laplacian and its eigendecomposition
2. Understand spectral graph convolutions and their connection to spatial methods
3. Implement and train Graph Attention Networks (GAT) using PyG
4. Build a complete graph classification pipeline with global pooling
5. Diagnose and understand the oversmoothing problem in deep GNNs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import KarateClub, TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool
from torch_geometric.utils import to_networkx

np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch: {torch.__version__}")

---

## 1. Graph Laplacian and Eigendecomposition

The **graph Laplacian** captures the structure of a graph and plays a central role in spectral graph theory.

### Definitions

- **Combinatorial Laplacian**: $L = D - A$
- **Normalized Laplacian**: $L_{\text{norm}} = I - D^{-1/2} A D^{-1/2}$
- **Random Walk Laplacian**: $L_{\text{rw}} = I - D^{-1} A$

Properties of $L$:
- Symmetric and positive semi-definite
- Smallest eigenvalue is 0 (with eigenvector $\mathbf{1}$)
- Number of zero eigenvalues = number of connected components
- The **Fiedler vector** (eigenvector of the second-smallest eigenvalue) reveals graph bipartitions

In [ ]:
# Create a graph with two loosely connected communities
G = nx.barbell_graph(8, 1)  # two cliques connected by a path

# Get adjacency and degree matrices
A = nx.adjacency_matrix(G).toarray().astype(np.float64)
n = A.shape[0]
D = np.diag(A.sum(axis=1))

# Compute different Laplacians
L = D - A  # Combinatorial Laplacian

D_inv_sqrt = np.diag(1.0 / np.sqrt(A.sum(axis=1)))
L_norm = np.eye(n) - D_inv_sqrt @ A @ D_inv_sqrt  # Normalized Laplacian

print(f"Graph: {n} nodes, {G.number_of_edges()} edges")
print(f"\nCombinatoral Laplacian L (first 5x5 block):")
print(L[:5, :5])
print(f"\nL is symmetric: {np.allclose(L, L.T)}")
print(f"L is positive semi-definite: {np.all(np.linalg.eigvalsh(L) >= -1e-10)}")

In [ ]:
# Eigendecomposition of the normalized Laplacian
eigenvalues, eigenvectors = np.linalg.eigh(L_norm)

print("Eigenvalues (sorted):")
print(np.round(eigenvalues, 6))
print(f"\nSmallest eigenvalue: {eigenvalues[0]:.10f} (should be ~0)")
print(f"Second smallest (algebraic connectivity): {eigenvalues[1]:.6f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Plot 1: Eigenvalue spectrum
axes[0].bar(range(len(eigenvalues)), eigenvalues, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Index')
axes[0].set_ylabel('Eigenvalue')
axes[0].set_title('Laplacian Spectrum')
axes[0].grid(True, alpha=0.3)

# Plot 2: Fiedler vector on the graph
pos = nx.spring_layout(G, seed=42)
fiedler = eigenvectors[:, 1]  # second eigenvector
nx.draw(G, pos, ax=axes[1], node_color=fiedler, cmap='RdBu',
        node_size=200, edge_color='lightgray', with_labels=True, font_size=7)
axes[1].set_title('Fiedler Vector (graph bipartition)')

# Plot 3: Third eigenvector
third_ev = eigenvectors[:, 2]
nx.draw(G, pos, ax=axes[2], node_color=third_ev, cmap='RdBu',
        node_size=200, edge_color='lightgray', with_labels=True, font_size=7)
axes[2].set_title('Third Eigenvector')

plt.tight_layout()
plt.show()

In [ ]:
# Spectral embedding: use the first k eigenvectors as node features
k = 3
spectral_coords = eigenvectors[:, 1:k+1]  # skip the first (constant) eigenvector

fig = plt.figure(figsize=(10, 5))

# 2D spectral embedding
ax1 = fig.add_subplot(121)
colors = ['steelblue' if i < 8 else ('gray' if i < 9 else 'coral')
          for i in range(n)]
ax1.scatter(spectral_coords[:, 0], spectral_coords[:, 1], c=colors,
            s=100, edgecolors='black', linewidths=0.5)
for i in range(n):
    ax1.annotate(str(i), (spectral_coords[i, 0], spectral_coords[i, 1]),
                 fontsize=7, ha='center', va='center')
ax1.set_xlabel('Eigenvector 2')
ax1.set_ylabel('Eigenvector 3')
ax1.set_title('2D Spectral Embedding')
ax1.grid(True, alpha=0.3)

# 3D spectral embedding
ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(spectral_coords[:, 0], spectral_coords[:, 1], spectral_coords[:, 2],
            c=colors, s=100, edgecolors='black', linewidths=0.5)
ax2.set_xlabel('EV 2')
ax2.set_ylabel('EV 3')
ax2.set_zlabel('EV 4')
ax2.set_title('3D Spectral Embedding')

plt.tight_layout()
plt.show()

print("Notice how the two cliques are clearly separated in spectral space!")

---

## 2. Spectral Graph Convolution

### Key Idea

In the spectral domain, graph convolution is pointwise multiplication:

$$x *_G g = U \, \text{diag}(\hat{g}) \, U^T x$$

where $U$ contains the eigenvectors of $L$ and $\hat{g}$ is the filter in the spectral domain.

### From Spectral to Spatial

- **ChebNet** (Defferrard et al., 2016): approximates spectral filters with Chebyshev polynomials, avoiding explicit eigendecomposition
- **GCN** (Kipf & Welling, 2017): first-order approximation of ChebNet, resulting in the spatial formula $\tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}XW$

In [ ]:
# Demonstrate spectral convolution on a signal

# Create a signal on the graph (e.g., indicator of one community)
signal = np.zeros(n)
signal[:8] = 1.0  # first clique

# Graph Fourier Transform: project signal onto eigenvectors
eigenvalues_comb, eigvecs_comb = np.linalg.eigh(L)
signal_hat = eigvecs_comb.T @ signal  # spectral coefficients

# Design different spectral filters
# Low-pass filter: keeps low-frequency (smooth) components
tau = 2.0  # controls cutoff
low_pass = np.exp(-tau * eigenvalues_comb)

# High-pass filter: keeps high-frequency (oscillatory) components
high_pass = 1.0 - low_pass

# Apply filters in spectral domain
signal_low = eigvecs_comb @ (low_pass * signal_hat)
signal_high = eigvecs_comb @ (high_pass * signal_hat)

# Visualize
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

# Original signal
nx.draw(G, pos, ax=axes[0], node_color=signal, cmap='coolwarm',
        node_size=200, edge_color='lightgray', with_labels=True, font_size=7,
        vmin=-0.5, vmax=1.5)
axes[0].set_title('Original Signal')

# Spectral coefficients
axes[1].bar(range(len(signal_hat)), signal_hat, color='steelblue', alpha=0.8)
axes[1].set_xlabel('Frequency index')
axes[1].set_ylabel('Coefficient')
axes[1].set_title('Graph Fourier Coefficients')
axes[1].grid(True, alpha=0.3)

# Low-pass filtered
nx.draw(G, pos, ax=axes[2], node_color=signal_low, cmap='coolwarm',
        node_size=200, edge_color='lightgray', with_labels=True, font_size=7)
axes[2].set_title('Low-pass Filtered (smooth)')

# High-pass filtered
nx.draw(G, pos, ax=axes[3], node_color=signal_high, cmap='coolwarm',
        node_size=200, edge_color='lightgray', with_labels=True, font_size=7)
axes[3].set_title('High-pass Filtered (edges)')

plt.tight_layout()
plt.show()

print("Low-pass filtering smooths the signal across the graph.")
print("High-pass filtering highlights transitions/boundaries.")
print("GCN effectively learns low-pass filters (why it smooths features).")

---

## 3. Graph Attention Networks (GAT)

**GAT** (Velickovic et al., 2018) replaces the fixed normalization of GCN with learned attention:

$$h_i' = \sigma\left(\sum_{j \in \mathcal{N}(i) \cup \{i\}} \alpha_{ij} W h_j\right)$$

where the attention coefficient $\alpha_{ij}$ is:

$$\alpha_{ij} = \frac{\exp(\text{LeakyReLU}(\mathbf{a}^T [Wh_i \| Wh_j]))}{\sum_{k \in \mathcal{N}(i) \cup \{i\}} \exp(\text{LeakyReLU}(\mathbf{a}^T [Wh_i \| Wh_k]))}$$

GAT also supports **multi-head attention** for stability.

In [ ]:
# Load the Karate Club dataset
dataset = KarateClub()
data = dataset[0]


class GAT(nn.Module):
    """Graph Attention Network for node classification."""

    def __init__(self, in_channels, hidden_channels, out_channels,
                 heads=4, dropout=0.6):
        super().__init__()
        # First GAT layer: multi-head attention
        # Output dim = hidden_channels * heads (concatenation of heads)
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads,
                             dropout=dropout)
        # Second GAT layer: single head, outputs class logits
        # Input dim = hidden_channels * heads (from concatenated output)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1,
                             concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x


# Initialize model
model_gat = GAT(
    in_channels=dataset.num_features,
    hidden_channels=8,
    out_channels=dataset.num_classes,
    heads=4,
    dropout=0.6,
)
print(model_gat)
print(f"\nTotal parameters: {sum(p.numel() for p in model_gat.parameters()):,}")

In [ ]:
# Train the GAT model
optimizer = torch.optim.Adam(model_gat.parameters(), lr=0.005, weight_decay=5e-4)
criterion = nn.CrossEntropyLoss()

gat_losses = []
gat_accs = []

model_gat.train()
for epoch in range(300):
    optimizer.zero_grad()
    out = model_gat(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()

    pred = out.argmax(dim=1)
    acc = (pred == data.y).float().mean().item()
    gat_losses.append(loss.item())
    gat_accs.append(acc)

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f} | Accuracy: {acc:.4f}")

print(f"\nFinal GAT accuracy: {gat_accs[-1]:.4f}")

In [ ]:
# Visualize attention weights
model_gat.eval()

# Get attention weights from the first GAT layer
# GATConv can return attention weights with return_attention_weights=True
with torch.no_grad():
    x_in = F.dropout(data.x, p=0.0, training=False)
    out1, (edge_idx, attn_weights) = model_gat.conv1(
        x_in, data.edge_index, return_attention_weights=True
    )

# Average attention across heads
# attn_weights shape: [num_edges, num_heads]
avg_attn = attn_weights.mean(dim=1).numpy()

# Visualize
G_karate = to_networkx(data, to_undirected=True)
pos_k = nx.spring_layout(G_karate, seed=42)

fig, ax = plt.subplots(figsize=(10, 7))

# Draw nodes colored by predicted class
with torch.no_grad():
    preds = model_gat(data.x, data.edge_index).argmax(dim=1).numpy()

# Build edge weight dict for visualization
edge_src = edge_idx[0].numpy()
edge_dst = edge_idx[1].numpy()
edge_weight_dict = {}
for i in range(len(edge_src)):
    s, d = int(edge_src[i]), int(edge_dst[i])
    key = (min(s, d), max(s, d))
    edge_weight_dict[key] = max(edge_weight_dict.get(key, 0), avg_attn[i])

# Get edge weights in the order of G_karate edges
edge_widths = []
for u, v in G_karate.edges():
    key = (min(u, v), max(u, v))
    edge_widths.append(edge_weight_dict.get(key, 0.01) * 5)

nx.draw(G_karate, pos_k, ax=ax, with_labels=True, node_color=preds,
        cmap='Set1', node_size=400, font_color='white', font_weight='bold',
        edge_color='gray', width=edge_widths, font_size=9)
ax.set_title('GAT Predictions with Attention Weights (edge thickness)', fontsize=14)
plt.tight_layout()
plt.show()

---

## 4. Graph Classification Pipeline

Unlike node classification, **graph classification** assigns a label to an entire graph.

Pipeline:
1. Learn node embeddings with GNN layers
2. **Pool** node embeddings into a single graph-level vector (global pooling)
3. Classify the graph-level vector with an MLP

We will use the **MUTAG** dataset: 188 molecules labeled as mutagenic or not.

In [ ]:
# Load the MUTAG dataset
mutag_dataset = TUDataset(root='/tmp/MUTAG', name='MUTAG')

print(f"Dataset: {mutag_dataset}")
print(f"Number of graphs: {len(mutag_dataset)}")
print(f"Number of features: {mutag_dataset.num_features}")
print(f"Number of classes: {mutag_dataset.num_classes}")

# Inspect a single graph
sample = mutag_dataset[0]
print(f"\nSample graph:")
print(f"  {sample}")
print(f"  Nodes: {sample.num_nodes}, Edges: {sample.num_edges}")
print(f"  Node features shape: {sample.x.shape}")
print(f"  Label: {sample.y.item()}")

# Visualize a few graphs
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for idx, ax in enumerate(axes):
    graph = mutag_dataset[idx]
    G_vis = to_networkx(graph, to_undirected=True)
    # Color nodes by their feature (argmax of one-hot)
    node_colors = graph.x.argmax(dim=1).numpy()
    pos = nx.spring_layout(G_vis, seed=42)
    nx.draw(G_vis, pos, ax=ax, node_color=node_colors, cmap='Set2',
            node_size=150, edge_color='gray', with_labels=False)
    ax.set_title(f'Graph {idx} (label={graph.y.item()})')
plt.tight_layout()
plt.show()

In [ ]:
# Train/test split
torch.manual_seed(42)
mutag_dataset = mutag_dataset.shuffle()

n_train = int(0.8 * len(mutag_dataset))
train_dataset = mutag_dataset[:n_train]
test_dataset = mutag_dataset[n_train:]

print(f"Training graphs: {len(train_dataset)}")
print(f"Test graphs: {len(test_dataset)}")

# Create data loaders (handles batching of different-sized graphs)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Inspect a batch
batch = next(iter(train_loader))
print(f"\nBatch: {batch}")
print(f"  batch.batch: assigns each node to its graph (shape: {batch.batch.shape})")
print(f"  Unique graphs in batch: {batch.batch.max().item() + 1}")

In [ ]:
class GraphClassifier(nn.Module):
    """GNN for graph classification with global mean pooling."""

    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=3):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()

        # First layer
        self.convs.append(GCNConv(in_channels, hidden_channels))
        self.bns.append(nn.BatchNorm1d(hidden_channels))

        # Hidden layers
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))
            self.bns.append(nn.BatchNorm1d(hidden_channels))

        # Last GNN layer
        self.convs.append(GCNConv(hidden_channels, hidden_channels))
        self.bns.append(nn.BatchNorm1d(hidden_channels))

        # Classification head (MLP)
        self.fc1 = nn.Linear(hidden_channels, hidden_channels)
        self.fc2 = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, batch):
        # Node-level: GNN layers
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=0.3, training=self.training)

        # Graph-level: Global mean pooling
        # Aggregates node features into a single vector per graph
        x = global_mean_pool(x, batch)  # [num_graphs, hidden_channels]

        # Classification MLP
        x = F.relu(self.fc1(x))
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.fc2(x)
        return x


model_gc = GraphClassifier(
    in_channels=mutag_dataset.num_features,
    hidden_channels=64,
    out_channels=mutag_dataset.num_classes,
    num_layers=3,
)
print(model_gc)
print(f"\nTotal parameters: {sum(p.numel() for p in model_gc.parameters()):,}")

In [ ]:
# Training and evaluation functions

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in loader:
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch.num_graphs
        pred = out.argmax(dim=1)
        correct += (pred == batch.y).sum().item()
        total += batch.num_graphs

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    for batch in loader:
        out = model(batch.x, batch.edge_index, batch.batch)
        pred = out.argmax(dim=1)
        correct += (pred == batch.y).sum().item()
        total += batch.num_graphs

    return correct / total

In [ ]:
# Training loop
optimizer = torch.optim.Adam(model_gc.parameters(), lr=0.01, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

train_losses = []
train_accs = []
test_accs = []

for epoch in range(100):
    loss, train_acc = train_epoch(model_gc, train_loader, optimizer, criterion)
    test_acc = evaluate(model_gc, test_loader)

    train_losses.append(loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {loss:.4f} | "
              f"Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")

print(f"\nBest test accuracy: {max(test_accs):.4f} (epoch {np.argmax(test_accs)+1})")

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, color='steelblue')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(train_accs, label='Train', color='steelblue')
ax2.plot(test_accs, label='Test', color='coral')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Graph Classification Accuracy (MUTAG)')
ax2.legend()
ax2.set_ylim(0, 1.05)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 5. Oversmoothing Demonstration

A critical issue with deep GNNs: as we stack more layers, node representations
converge to similar values, losing discriminative power. This is called **oversmoothing**.

**Why?** Each GCN layer averages over neighbors. After $k$ layers, each node's
representation captures its $k$-hop neighborhood. For many graphs, a few layers
are enough to reach all nodes, making all features converge.

Let us measure this empirically.

In [ ]:
class DeepGCN(nn.Module):
    """Variable-depth GCN to study oversmoothing."""

    def __init__(self, in_channels, hidden_channels, out_channels, num_layers):
        super().__init__()
        self.convs = nn.ModuleList()

        if num_layers == 1:
            self.convs.append(GCNConv(in_channels, out_channels))
        else:
            self.convs.append(GCNConv(in_channels, hidden_channels))
            for _ in range(num_layers - 2):
                self.convs.append(GCNConv(hidden_channels, hidden_channels))
            self.convs.append(GCNConv(hidden_channels, out_channels))

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x, edge_index)
            x = F.relu(x)
        x = self.convs[-1](x, edge_index)
        return x

    def get_all_layer_outputs(self, x, edge_index):
        """Returns intermediate representations for analysis."""
        outputs = [x.clone()]
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x, edge_index)
            x = F.relu(x)
            outputs.append(x.clone())
        x = self.convs[-1](x, edge_index)
        outputs.append(x.clone())
        return outputs

In [ ]:
# Train GCNs with different depths on Karate Club
data_kc = KarateClub()[0]

depths = [1, 2, 4, 8, 16, 32, 64]
results = {}

for depth in depths:
    torch.manual_seed(42)
    model_deep = DeepGCN(
        in_channels=data_kc.num_features,
        hidden_channels=16,
        out_channels=data_kc.y.max().item() + 1,
        num_layers=depth,
    )

    optimizer = torch.optim.Adam(model_deep.parameters(), lr=0.01, weight_decay=5e-4)
    criterion = nn.CrossEntropyLoss()

    best_acc = 0
    for epoch in range(200):
        model_deep.train()
        optimizer.zero_grad()
        out = model_deep(data_kc.x, data_kc.edge_index)
        loss = criterion(out[data_kc.train_mask], data_kc.y[data_kc.train_mask])
        loss.backward()
        optimizer.step()

        pred = out.argmax(dim=1)
        acc = (pred == data_kc.y).float().mean().item()
        best_acc = max(best_acc, acc)

    results[depth] = best_acc
    print(f"Depth {depth:2d} layers -> Best accuracy: {best_acc:.4f}")

In [ ]:
# Plot accuracy vs depth
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(list(results.keys()), list(results.values()), 'o-',
        color='steelblue', linewidth=2, markersize=8)
ax.set_xlabel('Number of GCN Layers', fontsize=12)
ax.set_ylabel('Best Accuracy', fontsize=12)
ax.set_title('Oversmoothing: Accuracy vs GCN Depth (Karate Club)', fontsize=13)
ax.set_xscale('log', base=2)
ax.set_xticks(depths)
ax.set_xticklabels(depths)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
ax.axhline(y=results[2], color='coral', linestyle='--', alpha=0.5,
           label=f'2-layer baseline ({results[2]:.2f})')
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Measure feature smoothness: how similar are node features after each layer?
# We use the Mean Average Distance (MAD) metric:
# MAD = average pairwise cosine distance between node features

def compute_mad(features: torch.Tensor) -> float:
    """Compute Mean Average Distance (cosine) between all node pairs."""
    # Normalize features
    norms = features.norm(dim=1, keepdim=True).clamp(min=1e-8)
    normalized = features / norms
    # Pairwise cosine similarity
    sim = normalized @ normalized.T
    n = sim.size(0)
    # Average off-diagonal distance (1 - similarity)
    mask = 1.0 - torch.eye(n)
    distances = (1.0 - sim) * mask
    mad = distances.sum() / mask.sum()
    return mad.item()


# Get a deep model and analyze layer-by-layer smoothness
torch.manual_seed(42)
deep_model = DeepGCN(
    in_channels=data_kc.num_features,
    hidden_channels=16,
    out_channels=data_kc.y.max().item() + 1,
    num_layers=32,
)

# Get outputs after each layer (untrained model is enough to show the effect)
deep_model.eval()
with torch.no_grad():
    layer_outputs = deep_model.get_all_layer_outputs(data_kc.x, data_kc.edge_index)

mad_values = [compute_mad(h) for h in layer_outputs]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(len(mad_values)), mad_values, 'o-', color='steelblue',
        linewidth=2, markersize=6)
ax.set_xlabel('Layer', fontsize=12)
ax.set_ylabel('Mean Average Distance (MAD)', fontsize=12)
ax.set_title('Feature Smoothness Across Layers (32-layer GCN)', fontsize=13)
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='red', linestyle='--', alpha=0.3, label='Fully smooth (MAD=0)')
ax.legend()

plt.tight_layout()
plt.show()

print(f"MAD after layer 0 (input):   {mad_values[0]:.6f}")
print(f"MAD after layer 2:           {mad_values[2]:.6f}")
print(f"MAD after layer 16:          {mad_values[16]:.6f}")
print(f"MAD after layer 32 (output): {mad_values[-1]:.6f}")
print("\nAs depth increases, MAD approaches 0 -> all nodes have similar features.")

---

## Summary

| Topic | Key Takeaway |
|-------|-------------|
| Graph Laplacian | $L = D - A$; eigenvalues reveal graph structure |
| Spectral convolution | Convolution = multiplication in the spectral domain; GCN is a first-order approx |
| GAT | Learned attention weights replace fixed normalization |
| Graph classification | Node embeddings + global pooling + MLP classifier |
| Oversmoothing | Deep GNNs converge node features; 2-4 layers often optimal |

**Next**: Day 3 will cover geometric data beyond graphs: manifolds, point clouds, and equivariance.

---

### Exercises

1. **Spectral clustering**: Use the Fiedler vector to partition the barbell graph. Compare with `nx.community.kernighan_lin_bisection`.
2. **GAT heads**: Try 1, 2, 4, 8 attention heads. Plot the accuracy vs number of heads.
3. **Pooling strategies**: Replace `global_mean_pool` with `global_max_pool` or `global_add_pool`. Which works best on MUTAG?
4. **Residual connections**: Add skip connections to the DeepGCN. Does it mitigate oversmoothing?